In [1]:
import pandas as pd
import os
from PIL import Image
from pathlib import Path
import json

from sklearn.model_selection import train_test_split
import torch
import torchvision.transforms as transforms

#to follow progres during training (train time)
from tqdm import tqdm

#for R CNN model
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

#for test of new image on pretrained model
import cv2

# and for iou
from collections import namedtuple
import numpy as np

#for maP
!pip install torchmetrics
from torchmetrics.detection.mean_ap import MeanAveragePrecision



In [12]:
#device cuda (compute unified device architecture) for google colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#pathway to drive file
#from google.colab import drive
#drive.mount('/content/drive')

#copy all to content, to avoid long loading time with drive mounting (once)
#import shutil
#shutil.copytree('/content/drive/MyDrive/Clouds-1000', '/content/Clouds-1000')

#resize all images before transform, too speed up training size
orig_path = Path('/content/Clouds-1000/all_images_and_labels/images')
small_path = Path('/content/Clouds-1000/all_images_and_labels/images_small')
small_path.mkdir(exist_ok=True)

for img_path in orig_path.glob('*.jpg'):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((224, 224))
    img.save(small_path / img_path.name)

dir_all = Path('/content/Clouds-1000/all_images_and_labels')


finished!


In [13]:
# split in test (30%) and train (70%) set
#create pd dataframe manually for images and labels
image_path=dir_all/'images_small'
mask_path=dir_all/'masks'

image_files=list(sorted(image_path.glob('*.jpg')))
mask_files=list(sorted(mask_path.glob('*.json')))

df=pd.DataFrame({'images': image_files, 'masks': mask_files})
# split into train, val, test sets: train_size=0.7, test_size=0.15 * 2
train_df, test_df=train_test_split(df, test_size=0.15, random_state=42)
train_df, val_df=train_test_split(train_df, test_size=0.15, random_state=42)

#check size
print(f"Set sizes:\nTrain:\t{len(train_df)}")  #70%
print(f"Val:\t{len(val_df)}")  #15%
print(f"Test:\t{len(test_df)}")  #15%

Set sizes:
Train:	719
Val:	127
Test:	150


In [14]:
#baseline model: faster R CNN
#https://medium.com/@RobuRishabh/understanding-and-implementing-faster-r-cnn-248f7b25ff96

# Load the pre-trained Faster R-CNN model with a ResNet-50 backbone
model_a = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

num_classes = 6  #number of classes (dataset classes + 1 for background)
in_features = model_a.roi_heads.box_predictor.cls_score.in_features #get number of input features for classifier
model_a.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) #replace model's head with new one (for number of classes in dataset)

#prepare data for R CNN
# Define transformations (equal to ResNet18 baseline) & resize images for smaller data
transform = transforms.Compose([ #withoug transforms.Resize((224, 224)), as done already before loading
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
]) #mean and std for generalized ImageNet dataset for RGB values

#create Custom Dataset to organise data in boxes/polygons and labels/classes
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.transform = transforms
        self.classes = ['Arvore', 'Estratocumuliformes', 'Cirriformes', 'Estratiformes', 'Cumuliformes']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):   #detect len for batches
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['images']).convert('RGB')
        orig_w, orig_h = img.size  #save original size (2592x1944)

        with open(self.df.iloc[idx]['masks']) as f:
            data = json.load(f)

        boxes, labels = [], []
        for obj in data['objects']:
            polygon = obj['points']['exterior']
            x_coords = [p[0] for p in polygon]
            y_coords = [p[1] for p in polygon]

            #to avoid unvalid boxes
            x1, y1, x2, y2 = min(x_coords), min(y_coords), max(x_coords), max(y_coords)
            if x2 <= x1 or y2 <= y1:
              continue
            boxes.append([min(x_coords), min(y_coords), max(x_coords), max(y_coords)])
            labels.append(self.class_to_idx[obj['classTitle']])

        #filter for empty boxes
        if len(boxes) == 0:
          boxes = [[0, 0, 1, 1]]
          labels = [0]

        # scale boxes to 224x224
        scale_x = 224 / orig_w
        scale_y = 224 / orig_h
        boxes = [[x1*scale_x, y1*scale_y, x2*scale_x, y2*scale_y] for x1,y1,x2,y2 in boxes]

        target = {}
        target["boxes"] = torch.tensor(boxes, dtype=torch.float32)
        target["labels"] = torch.tensor(labels, dtype=torch.int64)

        if self.transform is not None:
            img = self.transform(img)

        return img, target



In [15]:
# split into train, val, test sets: train_size=0.7, test_size=0.15 * 2
#load datasets
train_ds = CustomDataset(train_df, transforms=transform)
val_ds   = CustomDataset(val_df,   transforms=transform)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, #load parallel
                          pin_memory=True, #faster gpu
                          collate_fn=lambda x: tuple(zip(*x)))
valid_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, #load parallel
                          pin_memory=True, #faster gpu
                          collate_fn=lambda x: tuple(zip(*x)))

In [16]:
#training
model_a.to(device)

# Set up the optimizer
params = [p for p in model_a.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
num_epochs = 5
for epoch in range(num_epochs):
    model_a.train()
    train_loss = 0.0

    # Training loop
    #with loading bar
    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
      for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        # Forward pass
        loss_dict = model_a(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        losses.backward()
        optimizer.step()
        train_loss += losses.item()

      # Update learning rate
      lr_scheduler.step()
print(f'Epoch: {epoch + 1}, Loss: {train_loss / len(train_loader)}')
print("Training complete!")


Epoch 1/5:  28%|██▊       | 50/180 [1:34:42<4:06:14, 113.65s/it]


KeyboardInterrupt: 

In [ ]:
#list of all learnable parameters of r cnn model (model_a)
for name, param in model_a.named_parameters():
    print(name, param.shape)

In [ ]:
#evaluate training (without gradient calculation, only prediction: no_grad)
model_a.eval()
map_a=MeanAveragePrecision.to(device)
with torch.no_grad():
  for images, targets in valid_loader(): #targets optional as _, as it is not used here, only appearing in tupel (getitem)
    images=list(img.to(device) for img in images)
    prediction=model_a(images) #forward calculation through call
    preds_boxes=prediction[0]['boxes']
    true_boxes=targets[0]['boxes']
    #print bounding boxes for first images as example
    print(prediction[0]['boxes'])
    print(prediction[0]['labels'])

    #for maP change to cpu (not gpu compatible)
    true_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
    preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in predictions]

    map_a.update(preds_cpu, true_cpu)
result=map_a.compute()
print(f'Mean Average Prediction (maP score): {map_a}')



In [ ]:


#IoU:Intersection over Union to evaluate accuracy of boxes = area of overlap/total area of true & predicted box (=union))
#compare: iou>0.5 looks good (https://pyimagesearch.com/2016/11/07/intersection-over-union-iou-for-object-detection/)
#1. define the 'detection' object that stores the 3 attributes: folder, original box, predicted box
Detection = namedtuple("Detection", ["image_path", "true", "pred"])
#2. define function for intersection/overlapping
def bb_intersection_over_union(boxA, boxB):
	# determine (x, y)-coordinates of intersection rectangle
	xA = max(boxA[0], boxB[0])
	yA = max(boxA[1], boxB[1])
	xB = min(boxA[2], boxB[2])
	yB = min(boxA[3], boxB[3])
	#compute area of intersection rectangle
	interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
	#compute area of prediction and ground-truth (true) rectangles
	boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
	boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
	#compute intersection over union by taking intersection area and dividing by:
  #prediction + ground-truth areas - the intersection area
	iou_a = interArea / float(boxAArea + boxBArea - interArea)
	#return intersection over union value
	return iou_a


In [ ]:
#check trained model success for new image
#Load random image
img = Image.open("image_path/2021-06-03_15-46-00.jpeg")
# Apply same transformation as for training
img = transform(img)
img = img.unsqueeze(0).to(device)
#Model prediction
model.eval()
with torch.no_grad():
    prediction = model([img])
# Print the predicted bounding boxes and labels
print(prediction[0]['boxes'])
print(prediction[0]['labels'])

In [ ]:
#check how much training data is needed for success
